In [ ]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://afisha.yandex.ru/samara/events?page=40")

input() # как страница прогрузится нажать

source = driver.page_source

driver.quit()

In [ ]:
from bs4 import BeautifulSoup as BS
import json
soup = BS(source, features = "lxml")
for s in soup.find_all( class_= "events-list__item" ):
    print(s)
    j = json.loads(s.select_one(".event-card-react")["data-bem"])
    break

In [ ]:
from bs4 import BeautifulSoup as BS
import json
import dateparser

events = []
locations = {}

def location(lon, lat):
    return "latitude=%f;longitude=%f" % (lat, lon)

soup = BS(source, features = "lxml")
for s in soup.find_all( class_= "events-list__item" ):
    j = json.loads(s.select_one(".event-card-react")["data-bem"])
    props = j["event-card-react"]["props"]
    url = "https://afisha.yandex.ru" + props["link"]
    
    event = {
        "title": props["title"],
        "source_url": url,
        "description": props["title"],
        "price": props.get("ticketsPrice"),
        "cover_img_url": None,
        "location_id": None,
        "date": None
    }
    #print(props["type"])
    #print(props["tag"])
    if props.get("image"):
        if(props["image"].get("retina")):
            event["cover_img_url"] = props["image"]["retina"][list(props["image"]["retina"].keys())[-1]]
        else:
            event["cover_img_url"] = props["image"]["url"]

    #print(props["ageLimit"])
    if props.get("place") and props["place"].get("coordinates"):
        place = props["place"]
        place_id = int(place["id"], 16) % (1<<63)
        event["location_id"] = place_id
        
        if not locations.get(place_id):
            loc = {
                "id": place_id,
                "title": place["title"],
                "address": place["address"],
            }

            if place.get("coordinates"):
                loc.update(place["coordinates"])
            else:
                loc.update({"latitude": None, "longitude": None})
            #todo: city

            locations[place_id] = loc
    else:
        event["location_id"] = None
    
    if props.get("additionalInfo"):
        date = dateparser.parse(props["additionalInfo"])
        if date:
            event["date"] = str(date)

    events.append(event)



In [ ]:
def wrap(item):
    if isinstance(item, str):
        return "'%s'" % item
    if item is None:
        return "NULL"
    return str(item)

def wrapper(container):
    return "(%s)" % ",\n".join(list(map(wrap, container)))

def dbPopulator(filename, table_name, data):
    f = open(filename, "w", encoding="utf-8")

    kys = next(iter(data)).keys()

    f.write(f"INSERT INTO {table_name} ")

    f.write("(%s)" % ",".join(kys))

    f.write("\nVALUES ")

    f.write(",\n".join([wrapper(d.values()) for d in data]))

    f.write(";")

    f.close()

dbPopulator("1populate_locations.sql", "locations", locations.values())
dbPopulator("2populate_events.sql", "Events", events)